Step 1: We import necessary modules and upload posture files and session files
we also make a sanity-check using their info

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

posture = pd.read_csv("../data/processed/posture_analysis_cleaned.csv")
sessions = pd.read_csv("../data/processed/sessions_cleaned.csv")

posture.info()
sessions.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7000 entries, 0 to 6999
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   id          7000 non-null   object 
 1   user_id     7000 non-null   object 
 2   session_id  7000 non-null   object 
 3   timestamp   7000 non-null   object 
 4   metrics     7000 non-null   object 
 5   posture     7000 non-null   object 
 6   confidence  7000 non-null   float64
dtypes: float64(1), object(6)
memory usage: 382.9+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   id          1000 non-null   object
 1   user_id     1000 non-null   object
 2   started_at  1000 non-null   object
 3   ended_at    996 non-null    object
 4   status      1000 non-null   object
 5   created_at  1000 non-null   object
 6   updated_at  1000 non-null   object
 7   a

Before Merging we have some work to do
We must change the type of certain columns to datetime type to make it more convenient later on

In [2]:
posture["timestamp"] = pd.to_datetime(posture["timestamp"], errors="coerce")

sessions["started_at"] = pd.to_datetime(sessions["started_at"], errors="coerce")
sessions["ended_at"] = pd.to_datetime(sessions["ended_at"], errors="coerce")
sessions["created_at"] = pd.to_datetime(sessions["created_at"], errors="coerce")
sessions["updated_at"] = pd.to_datetime(sessions["updated_at"], errors="coerce")


We clean the columns which may be used as an index for merging

In [3]:
posture["id"] = posture["id"].astype(str).str.strip()
posture["user_id"] = posture["user_id"].astype(str).str.strip()
posture["session_id"] = posture["session_id"].astype(str).str.strip()

sessions["id"] = sessions["id"].astype(str).str.strip()
sessions["user_id"] = sessions["user_id"].astype(str).str.strip()


We once again clean the index columns used to merge for sanity purposes
we then choose the columns we wish to bring over because other data is not relevant to our purpose
Finally we merge postures and sessions

In [4]:
# 0) Ensure consistent dtypes/whitespace for keys
posture["session_id"] = posture["session_id"].astype(str).str.strip()
sessions["id"]        = sessions["id"].astype(str).str.strip()

# 1) Sanity: is sessions.id a primary key? (no duplicates)
dup_sessions = sessions["id"].duplicated(keep=False).sum()
print("Duplicate session IDs in sessions:", dup_sessions)

# If duplicates exist, pick the “latest” record per session (example rule):
if dup_sessions:
    sessions = (sessions
                .sort_values(["id","updated_at"], na_position="last")
                .drop_duplicates(subset=["id"], keep="last"))

# 2) (Optional) Select only the columns you want to bring over
sess_cols = ["id","user_id","started_at","ended_at","status","created_at","updated_at"]
sess_view = sessions.loc[:, [c for c in sess_cols if c in sessions.columns]]

# 3) Merge with validation + indicator + explicit suffixes
merged = posture.merge(
    sess_view,
    left_on="session_id",
    right_on="id",
    how="left",
    validate="many_to_one",         # posture→sessions should be many-to-one
    indicator=True,
    suffixes=("_posture", "_session")
)


Duplicate session IDs in sessions: 0


In [5]:
merged.head()

,id_posture,user_id_posture,session_id,timestamp,metrics,posture,confidence,id_session,user_id_session,started_at,ended_at,status,created_at,updated_at,_merge
0,2cdc08d3-f80d-49a9-84e6-9dca27d22ba6,1ac653f4-5275-4e21-814d-56992851e16a,296056ea-9e62-4852-875a-8269dbea4c75,2025-10-06 08:22:32.903048+00:00,"{""cva_angle"": 1.9842126808714795, ""head_tilt"":...",good,1.00,296056ea-9e62-4852-875a-8269dbea4c75,1ac653f4-5275-4e21-814d-56992851e16a,2025-10-06 08:19:45.155000+00:00,2025-10-06 08:24:36.030000+00:00,ended,2025-10-06 08:19:45.154915,2025-10-06 08:24:36.030,both
1,4bc283aa-000f-4ba6-a93d-f23f4cb028a7,1ac653f4-5275-4e21-814d-56992851e16a,296056ea-9e62-4852-875a-8269dbea4c75,2025-10-06 08:22:27.862245+00:00,"{""cva_angle"": -6.738489703970799, ""head_tilt"":...",fair,0.75,296056ea-9e62-4852-875a-8269dbea4c75,1ac653f4-5275-4e21-814d-56992851e16a,2025-10-06 08:19:45.155000+00:00,2025-10-06 08:24:36.030000+00:00,ended,2025-10-06 08:19:45.154915,2025-10-06 08:24:36.030,both
2,bf5e458f-f5c5-4aaf-9f9f-7f5ced2ef2f3,1ac653f4-5275-4e21-814d-56992851e16a,296056ea-9e62-4852-875a-8269dbea4c75,2025-10-06 08:22:22.786503+00:00,"{""cva_angle"": 0.12198708517803425, ""head_tilt""...",good,1.00,296056ea-9e62-4852-875a-8269dbea4c75,1ac653f4-5275-4e21-814d-56992851e16a,2025-10-06 08:19:45.155000+00:00,2025-10-06 08:24:36.030000+00:00,ended,2025-10-06 08:19:45.154915,2025-10-06 08:24:36.030,both
3,7d27716e-fcf1-40d4-b3b5-659c9f2944cf,1ac653f4-5275-4e21-814d-56992851e16a,296056ea-9e62-4852-875a-8269dbea4c75,2025-10-06 08:22:17.814075+00:00,"{""cva_angle"": 1.2171196604729175, ""head_tilt"":...",good,1.00,296056ea-9e62-4852-875a-8269dbea4c75,1ac653f4-5275-4e21-814d-56992851e16a,2025-10-06 08:19:45.155000+00:00,2025-10-06 08:24:36.030000+00:00,ended,2025-10-06 08:19:45.154915,2025-10-06 08:24:36.030,both
4,23342e31-0dbf-46f9-a94b-b7ba5fcb3806,1ac653f4-5275-4e21-814d-56992851e16a,296056ea-9e62-4852-875a-8269dbea4c75,2025-10-06 08:22:12.681625+00:00,"{""cva_angle"": 2.1016236763927907, ""head_tilt"":...",good,1.00,296056ea-9e62-4852-875a-8269dbea4c75,1ac653f4-5275-4e21-814d-56992851e16a,2025-10-06 08:19:45.155000+00:00,2025-10-06 08:24:36.030000+00:00,ended,2025-10-06 08:19:45.154915,2025-10-06 08:24:36.030,both


In [6]:
merged.to_csv(r"c:\Users\niran\OneDrive\Desktop\SitSense\data\merged\posture_sessions_merged.csv", index=False)
print("Saved merged dataset to data/merged/posture_sessions_merged.csv")

Saved merged dataset to data/merged/posture_sessions_merged.csv
